In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_fasta_snapshot as ncbi_fasta_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_fasta_snapshot_module = importlib.reload(ncbi_fasta_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_ncbi_protein_fasta_snapshot = (
    ncbi_fasta_snapshot_module.resolve_ncbi_protein_fasta_snapshot
)

from src.pago_pipeline.storage import sha256_of_file

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\pago-proj\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Define FASTA snapshot configuration
# =============================================================================

XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)
FASTA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_fasta"
)
FASTA_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
SEQUENCE_LINE_WIDTH = 60
UPDATE_LATEST_DIRECTORY = True

print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"FASTA snapshot root directory: {FASTA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"FASTA snapshot mode: {FASTA_SNAPSHOT_MODE}")
print(f"Sequence line width: {SEQUENCE_LINE_WIDTH}")

XML snapshot root directory: C:\pago-proj\pAgo-project\data\01-raw\protein_xml_snapshots
Metadata snapshot root directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv
FASTA snapshot root directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta
FASTA snapshot mode: reuse_latest_or_create
Sequence line width: 60


In [4]:
# =============================================================================
# CELL 4 — Resolve active FASTA snapshot
# =============================================================================

fasta_snapshot_payload = resolve_ncbi_protein_fasta_snapshot(
    snapshot_mode=FASTA_SNAPSHOT_MODE,
    snapshot_root_directory=FASTA_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    sequence_line_width=SEQUENCE_LINE_WIDTH,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

protein_fasta_snapshot_directory = fasta_snapshot_payload["snapshot_directory"]
protein_fasta_manifest_file_path = fasta_snapshot_payload["manifest_file_path"]
protein_fasta_file_path = fasta_snapshot_payload["fasta_file_path"]
protein_fasta_manifest_payload = fasta_snapshot_payload["manifest"]
protein_fasta_output_directory = protein_fasta_snapshot_directory

protein_fasta_file_sha256 = sha256_of_file(input_file_path=protein_fasta_file_path)
protein_fasta_manifest_file_sha256 = sha256_of_file(
    input_file_path=protein_fasta_manifest_file_path,
)

source_metadata_snapshot_relative_path = protein_fasta_manifest_payload.get(
    "source_metadata_snapshot_relative_path"
)
if (
    not isinstance(source_metadata_snapshot_relative_path, str)
    or not source_metadata_snapshot_relative_path
):
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_metadata_snapshot_relative_path."
    )

source_metadata_csv_file_name = protein_fasta_manifest_payload.get(
    "source_metadata_csv_file_name"
)
if not isinstance(source_metadata_csv_file_name, str) or not source_metadata_csv_file_name:
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_metadata_csv_file_name."
    )

source_metadata_snapshot_directory = (
    METADATA_SNAPSHOT_ROOT_DIRECTORY / source_metadata_snapshot_relative_path
)
source_metadata_csv_file_path = (
    source_metadata_snapshot_directory / source_metadata_csv_file_name
)

source_xml_snapshot_relative_path = protein_fasta_manifest_payload.get(
    "source_xml_snapshot_relative_path"
)
if not isinstance(source_xml_snapshot_relative_path, str) or not source_xml_snapshot_relative_path:
    raise RuntimeError(
        "FASTA snapshot manifest is missing source_xml_snapshot_relative_path."
    )

source_xml_snapshot_directory = (
    XML_SNAPSHOT_ROOT_DIRECTORY / source_xml_snapshot_relative_path
)

print("Resolved FASTA snapshot successfully.")
print(f"FASTA snapshot directory: {protein_fasta_snapshot_directory}")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"FASTA manifest file path: {protein_fasta_manifest_file_path}")
print(f"Source metadata CSV file path: {source_metadata_csv_file_path}")

Latest FASTA snapshot is available. Reusing frozen snapshot.
Resolved FASTA snapshot successfully.
FASTA snapshot directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest
FASTA file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\protein_sequences.fasta
FASTA manifest file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\manifest.json
Source metadata CSV file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\snapshots\2026-05-19T19-55-24Z__q_891f443d754c\protein_metadata.csv


In [5]:
# =============================================================================
# CELL 5 — Print FASTA snapshot summary
# =============================================================================

print("Protein FASTA snapshot is ready.")
print(
    f"Snapshot created at UTC: {protein_fasta_manifest_payload['snapshot_created_at_utc']}"
)
print(f"Rows read from source metadata CSV: {protein_fasta_manifest_payload['row_count']}")
print(f"FASTA records written: {protein_fasta_manifest_payload['fasta_record_count']}")
print(
    "Rows skipped because the AA sequence column was empty: "
    f"{protein_fasta_manifest_payload['skipped_missing_sequence_count']}"
)
print(
    "Source metadata snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_metadata_snapshot_relative_path']}"
)

Protein FASTA snapshot is ready.
Snapshot created at UTC: 2026-05-25T17:59:53Z
Rows read from source metadata CSV: 41345
FASTA records written: 41345
Rows skipped because the AA sequence column was empty: 0
Source metadata snapshot relative path: snapshots\2026-05-19T19-55-24Z__q_891f443d754c


In [6]:
# =============================================================================
# CELL 6 — Print FASTA artifact summary
# =============================================================================

print("Persisted FASTA snapshot artifacts:")
print(f"FASTA snapshot directory: {protein_fasta_snapshot_directory}")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"FASTA file SHA-256: {protein_fasta_file_sha256}")
print(f"FASTA manifest file path: {protein_fasta_manifest_file_path}")
print(f"FASTA manifest SHA-256: {protein_fasta_manifest_file_sha256}")
print(f"Source metadata snapshot directory: {source_metadata_snapshot_directory}")
print(f"Source metadata CSV file path: {source_metadata_csv_file_path}")
print(
    "Source metadata CSV SHA-256: "
    f"{protein_fasta_manifest_payload['source_metadata_csv_file_sha256']}"
)
print(
    "Source metadata manifest SHA-256: "
    f"{protein_fasta_manifest_payload['source_metadata_manifest_sha256']}"
)

Persisted FASTA snapshot artifacts:
FASTA snapshot directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest
FASTA file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\protein_sequences.fasta
FASTA file SHA-256: 47925c313069e5f95d59bf386611694f706f63ffe18e1c2e90d4b8e89d5e9c90
FASTA manifest file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_fasta\latest\manifest.json
FASTA manifest SHA-256: 86d30db804a56e8c1f636dc52ad4633db766e272c4d0ad520bd146729336ba40
Source metadata snapshot directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\snapshots\2026-05-19T19-55-24Z__q_891f443d754c
Source metadata CSV file path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\snapshots\2026-05-19T19-55-24Z__q_891f443d754c\protein_metadata.csv
Source metadata CSV SHA-256: 80977551ffd3a005e7d55005ff8cc4e07da28058df3d6c99e99caa2b2edaf563
Source metadata manifest SHA-256: f85fd05ce74f6397f6bad58763e9c0ea1

In [7]:
# =============================================================================
# CELL 7 — Inspect metadata columns used for FASTA export
# =============================================================================

fasta_source_columns = [
    "protein_uid",
    "gbseq__accession_version",
    "gbseq__length",
    "gbseq__organism",
    "gbseq__definition",
    "gbseq__sequence",
]
preview_row_limit = 3

fasta_source_preview_dataframe = pd.read_csv(
    source_metadata_csv_file_path,
    usecols=fasta_source_columns,
    nrows=preview_row_limit,
)

fasta_source_preview_dataframe

,protein_uid,gbseq__accession_version,gbseq__definition,gbseq__length,gbseq__organism,gbseq__sequence
0,1000250755,KXK13845.1,MAG: hypothetical protein UZ15_CFX003003232 [C...,709,Chloroflexi bacterium OLB15,mqassayqpfvevfpvkqenigplsayklilkannesertalerki...
1,1000266463,KXK28958.1,MAG: hypothetical protein UZ01_02401 [Candidat...,1043,Candidatus Brocadia sinica,mkridikeflrsftvrpngalnvflgagasvqagiptagmliwqfk...
2,1000285434,KXK47085.1,MAG: hypothetical protein UZ10_BCD003000691 [B...,365,Bacteroidetes bacterium OLB10,mkaeyiqepfllfgkgksicpregiselsvydtviearknqlllgi...


In [8]:
# =============================================================================
# CELL 8 — Preview FASTA output
# =============================================================================

preview_line_limit = 8
fasta_preview_lines: list[str] = []

with protein_fasta_file_path.open("r", encoding="utf-8") as fasta_file_handle:
    for _ in range(preview_line_limit):
        fasta_line = fasta_file_handle.readline()
        if not fasta_line:
            break
        fasta_preview_lines.append(fasta_line.rstrip("\n"))

print("Preview of the exported multi-FASTA file:")
print("\n".join(fasta_preview_lines))

Preview of the exported multi-FASTA file:
>protein_uid=1000250755|accession=KXK13845.1|length=709|organism=Chloroflexi_bacterium_OLB15 MAG: hypothetical protein UZ15_CFX003003232 [Chloroflexi bacterium OLB15]
mqassayqpfvevfpvkqenigplsayklilkannesertalerkiggkltyrlggilpg
twvwsngrvitdtpahpiklvmqieelrasfdlyaplegveevygwaadsatisdfvvrg
siekiaseiqqalaktpsnipnvrvdrefrarswvvngqpaislatnsrliyesdieqya
enqdkitdliglgvidrtsslqgeiikvvgqlgeqrerllsltqreemadilanapdtrw
vlriqagrneydyvsdalaiiirledverfgidpahaeralhlkpaqraqmvkavsdvvk
ahgyigdaytlnnapalfsnaapkvrlrfggsktrdyvethlmadfqasgafklrggtte
pppplrinlinaasdtvddfleamrrqcerdfnqrleivrerkmrvasranlesavrllt


In [9]:
# =============================================================================
# CELL 9 — Print FASTA provenance summary
# =============================================================================

print("FASTA provenance summary:")
print(
    "Source metadata snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_metadata_snapshot_relative_path']}"
)
print(
    "Source XML snapshot relative path: "
    f"{protein_fasta_manifest_payload['source_xml_snapshot_relative_path']}"
)
print(
    "Source XML snapshot directory name: "
    f"{protein_fasta_manifest_payload['source_xml_snapshot_directory_name']}"
)
print(f"Search query: {protein_fasta_manifest_payload['search_query']}")
print(f"Translated query: {protein_fasta_manifest_payload['translated_query']}")

FASTA provenance summary:
Source metadata snapshot relative path: snapshots\2026-05-19T19-55-24Z__q_891f443d754c
Source XML snapshot relative path: snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Source XML snapshot directory name: 2026-04-09T00-51-02Z__q_891f443d754c
Search query: PIWI[All Fields] AND Bacteria[Organism]
Translated query: PIWI[All Fields] AND ("Bacteria"[Organism] OR "Bacteria Latreille et al. 1825"[Organism])


In [10]:
# =============================================================================
# CELL 10 — Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- protein_fasta_snapshot_directory")
print("- protein_fasta_output_directory")
print("- protein_fasta_file_path")
print("- protein_fasta_manifest_file_path")
print("- protein_fasta_manifest_payload")
print("- source_metadata_snapshot_directory")
print("- source_metadata_csv_file_path")
print("- source_xml_snapshot_directory")

Variables exposed for downstream notebooks:
- protein_fasta_snapshot_directory
- protein_fasta_output_directory
- protein_fasta_file_path
- protein_fasta_manifest_file_path
- protein_fasta_manifest_payload
- source_metadata_snapshot_directory
- source_metadata_csv_file_path
- source_xml_snapshot_directory
